# 04. Data Cleaning — Sanitization, Type Coercion & Schema Validation

Real-world datasets arrive contaminated with inconsistent casing, leading/trailing whitespace, non-standardized datetime strings, dirty currency symbols, and out-of-bounds records.

---

### Core Data Cleaning Operations
1. **String Sanitization**: Trimming whitespace, standardizing case, stripping regex patterns.
2. **Type Coercion**: Converting currency strings to floats and parsing mixed date formats.
3. **Domain Constraint Enforcement**: Capping negative ages, invalid percentages, and future registration dates.

In [1]:
import numpy as np
import pandas as pd

# Synthesize realistic dirty enterprise customer data
raw_dirty_data = {
    'cust_name': ['  Alice Johnson ', 'BOB SMITH', 'charlie brown  ', 'Diana Prince', '  Evan Wright  '],
    'signup_date': ['2023-01-15', '12/04/2022', '2023/07/20', 'Jan 10, 2021', '2022-11-30'],
    'monthly_spend': ['$1,250.50', ' $450.00 ', '$2,100.75', '$98.20', '$3,410.00'],
    'age': [29, -5, 45, 135, 38],  # -5 and 135 are out-of-domain
    'state_code': ['ca', ' NY ', 'Texas', 'FL', 'ca ']
}

df_dirty = pd.DataFrame(raw_dirty_data)
print("Raw Contaminated Dataframe:")
print(df_dirty)

Raw Contaminated Dataframe:
         cust_name   signup_date monthly_spend  age state_code
0   Alice Johnson     2023-01-15     $1,250.50   29         ca
1        BOB SMITH    12/04/2022       $450.00   -5        NY 
2  charlie brown      2023/07/20     $2,100.75   45      Texas
3     Diana Prince  Jan 10, 2021        $98.20  135         FL
4      Evan Wright    2022-11-30     $3,410.00   38        ca 

## 1. Production Vectorized Cleaning Pipeline

In [1]:
df_clean = df_dirty.copy()

# 1. Clean names: strip whitespace and title-case
df_clean['cust_name'] = df_clean['cust_name'].str.strip().str.title()

# 2. Parse heterogeneous datetime formats
df_clean['signup_date'] = pd.to_datetime(df_clean['signup_date'], format='mixed')

# 3. Strip currency symbols, commas, and convert to float
df_clean['monthly_spend'] = (
    df_clean['monthly_spend']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(float)
)

# 4. Standardize state codes with dictionary mapping
state_map = {'ca': 'CA', 'ny': 'NY', 'texas': 'TX', 'fl': 'FL'}
df_clean['state_code'] = (
    df_clean['state_code']
    .str.strip()
    .str.lower()
    .map(lambda x: state_map.get(x, x.upper()))
)

# 5. Domain constraint validation on Age: coerce invalid ages to NaN or boundaries
df_clean['age'] = df_clean['age'].apply(lambda a: a if 18 <= a <= 100 else np.nan)

print("Sanitized and Validated Dataframe:")
print(df_clean)
print("\nData Types After Cleaning:")
print(df_clean.dtypes)

Sanitized and Validated Dataframe:
       cust_name signup_date  monthly_spend    age state_code
0  Alice Johnson  2023-01-15      1,250.500 29.000         CA
1      Bob Smith  2022-12-04        450.000    NaN         NY
2  Charlie Brown  2023-07-20      2,100.750 45.000         TX
3   Diana Prince  2021-01-10         98.200    NaN         FL
4    Evan Wright  2022-11-30      3,410.000 38.000         CA

Data Types After Cleaning:
cust_name                object
signup_date      datetime64[ns]
monthly_spend           float64
age                     float64
state_code               object
dtype: object

## 2. Key Cleaning Principles
- **Idempotence**: Running the cleaning script twice should yield the exact same result.
- **Fail Gracefully**: Use `pd.to_numeric(errors='coerce')` and `pd.to_datetime(errors='coerce')` to prevent crashes on rogue characters.
- **Audit Trails**: Always retain an audit record or log of the count of modified or coerced records.